In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, subprocess, sys, time
import numpy as np
import pandas as pd

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
CASE_ROWS = [5192, 40099, 63018]

if importlib.util.find_spec("numpyro") is None:
    print("Installing NumPyro without changing Kaggle's JAX/CUDA packages...")
    subprocess.run([sys.executable,"-m","pip","install","--quiet","--no-deps","numpyro==0.22.0"],check=True)

import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from numpyro.infer.initialization import init_to_value
from numpyro.diagnostics import summary as numpyro_summary

print("JAX backend:",jax.default_backend(),"| devices:",jax.devices(),"| NumPyro:",numpyro.__version__)
if jax.default_backend() != "gpu":
    raise RuntimeError("GPU preflight failed: set Kaggle Accelerator to GPU T4 x2, restart the session, then Run All.")

sources=[]
for mp in INPUT.rglob("simulation_metadata.json"):
    if mp.parent.name!="binary": continue
    try: meta=json.loads(mp.read_text(encoding="utf-8"))
    except Exception: continue
    if meta.get("family")=="binary" and int(meta.get("parameter_draws_unique",0))>=100_000:
        sources.append((meta.get("created_utc",""),mp.parent))
if not sources:
    raise FileNotFoundError("Attach Day 7 Cosmic swan Bayesian output with binary simulation files.")
_,BINARY_DIR=sorted(sources,key=lambda x:x[0])[-1]
observations=np.load(BINARY_DIR/"observations.npy",mmap_mode="r")
parameters=pd.read_csv(BINARY_DIR/"parameters.csv")
time_days=np.asarray(np.load(BINARY_DIR/"time_days.npy"),dtype=float)

map_paths=list(INPUT.rglob("day9_4_boundary_parameter_recovery.csv"))
if not map_paths:
    raise FileNotFoundError("Attach Day 9.4 output containing day9_4_boundary_parameter_recovery.csv.")
map_path=max(map_paths,key=lambda p:p.stat().st_mtime)
map_df=pd.read_csv(map_path)
needed={"simulation_row","parameter","map_estimate","known_truth"}
if not needed.issubset(map_df.columns):
    raise ValueError(f"Day 9.4 parameter table is missing: {sorted(needed-set(map_df.columns))}")
if any(row>=len(parameters) for row in CASE_ROWS):
    raise ValueError("A selected simulation row is absent from Day 7 input.")

OUT=WORK/("day9-5-targeted-nuts-"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUT.mkdir(parents=True,exist_ok=False)
CHAIN_METHOD="parallel" if len(jax.devices())>=2 else "sequential"
WARMUP,SAMPLES,CHAINS=500,800,2
print("[PASS] Day 7 input:",BINARY_DIR)
print("[PASS] Day 9.4 input:",map_path)
print("[PASS] Selected cases:",CASE_ROWS,"| output:",OUT)
print("NUTS:",CHAINS,"chains x",WARMUP,"warmup +",SAMPLES,"draws; chain method:",CHAIN_METHOD)


In [ ]:
def smooth_clip(x,sharpness=500.0):
    return (jax.nn.softplus(sharpness*x)-jax.nn.softplus(sharpness*(x-1.0)))/sharpness

def predict_jax(t,period,phase,depth,ratio,duration,ellip,reflection):
    orb=jnp.mod(t/period-phase+0.5,1.0)-0.5
    sec=jnp.mod(orb,1.0)-0.5
    half=jnp.maximum(duration/2.0,1e-6)
    ingress=jnp.maximum(half*0.18,1e-6)
    primary=smooth_clip((half+ingress-jnp.abs(orb))/(2.0*ingress))
    secondary=smooth_clip((half+ingress-jnp.abs(sec))/(2.0*ingress))
    return 1.0-depth*primary-depth*ratio*secondary+ellip*jnp.cos(4*jnp.pi*orb)+reflection*jnp.cos(2*jnp.pi*orb)

def scalar_metric(diagnostics,key):
    vals=[]
    for item in diagnostics.values():
        if key in item:
            a=np.asarray(item[key])
            if a.size==1 and np.isfinite(a).all(): vals.append(float(a.reshape(-1)[0]))
    return vals

case_summaries,interval_rows,posterior_draws,plot_rows=[],[],{},[]
for case_no,row in enumerate(CASE_ROWS,1):
    started=time.time()
    subset=map_df[map_df["simulation_row"].astype(int)==int(row)]
    if subset.empty: raise ValueError(f"No Day 9.4 MAP estimates for row {row}.")
    map_values=subset.set_index("parameter")["map_estimate"].astype(float).to_dict()
    required_params={"period_days","phase","depth_primary","depth_ratio","duration_fraction",
                     "ellipsoidal_amplitude","reflection_amplitude","noise_sigma"}
    if not required_params.issubset(map_values):
        raise ValueError(f"Day 9.4 row {row} lacks estimates: {sorted(required_params-set(map_values))}")

    t=np.asarray(time_days,float)
    y=np.asarray(observations[int(row)],float)
    ok=np.isfinite(t)&np.isfinite(y); t,y=t[ok],y[ok]
    order=np.argsort(t); t,y=t[order],y[order]
    y=y/np.median(y); t=t-t.min()
    period0=float(map_values["period_days"])
    period_width=max(0.15,0.02*period0)
    p_low,p_high=max(0.1,period0-period_width),min(13.0,period0+period_width)
    phase0=float(np.mod(map_values["phase"],1.0))
    observed_j,time_j=jnp.asarray(y),jnp.asarray(t)

    def model():
        period=numpyro.sample("period_days",dist.Uniform(p_low,p_high))
        phase_offset=numpyro.sample("phase_offset",dist.Normal(0.0,0.025))
        depth=numpyro.sample("depth_primary",dist.Uniform(0.002,0.45))
        ratio=numpyro.sample("depth_ratio",dist.Uniform(0.0,1.0))
        duration=numpyro.sample("duration_fraction",dist.Uniform(0.005,0.12))
        ellip=numpyro.sample("ellipsoidal_amplitude",dist.Uniform(0.0,0.02))
        reflection=numpyro.sample("reflection_amplitude",dist.Uniform(-0.01,0.01))
        log_noise=numpyro.sample("log_noise",dist.Uniform(np.log(1e-4),np.log(0.03)))
        phase=jnp.mod(phase0+phase_offset,1.0)
        pred=predict_jax(time_j,period,phase,depth,ratio,duration,ellip,reflection)
        numpyro.sample("flux",dist.Normal(pred,jnp.exp(log_noise)),obs=observed_j)

    init={
        "period_days":float(np.clip(period0,p_low+1e-6,p_high-1e-6)),
        "phase_offset":0.0,
        "depth_primary":float(np.clip(map_values["depth_primary"],0.002001,0.449999)),
        "depth_ratio":float(np.clip(map_values["depth_ratio"],1e-5,0.99999)),
        "duration_fraction":float(np.clip(map_values["duration_fraction"],0.005001,0.119999)),
        "ellipsoidal_amplitude":float(np.clip(map_values["ellipsoidal_amplitude"],1e-5,0.01999)),
        "reflection_amplitude":float(np.clip(map_values["reflection_amplitude"],-0.00999,0.00999)),
        "log_noise":float(np.log(np.clip(map_values["noise_sigma"],1.0001e-4,0.02999))),
    }
    kernel=NUTS(model,init_strategy=init_to_value(values=init),target_accept_prob=0.95,
                max_tree_depth=10,dense_mass=True)
    mcmc=MCMC(kernel,num_warmup=WARMUP,num_samples=SAMPLES,num_chains=CHAINS,
              chain_method=CHAIN_METHOD,progress_bar=True)
    try:
        mcmc.run(jax.random.PRNGKey(95000+int(row)))
        grouped=mcmc.get_samples(group_by_chain=True)
        flat=mcmc.get_samples(group_by_chain=False)
        diagnostic=numpyro_summary(grouped,group_by_chain=True)
        rhat=scalar_metric(diagnostic,"r_hat")
        ess=scalar_metric(diagnostic,"n_eff")
        divergences=int(np.asarray(mcmc.get_extra_fields(group_by_chain=True)["diverging"]).sum())
        max_rhat=max(rhat) if rhat else float("nan")
        min_ess=min(ess) if ess else float("nan")
        status="screen_pass" if np.isfinite(max_rhat) and np.isfinite(min_ess) and max_rhat<=1.05 and min_ess>=100 and divergences==0 else "needs_review"
        transformed={k:np.asarray(v) for k,v in flat.items()}
        transformed["phase"]=(phase0+transformed["phase_offset"])%1.0
        transformed["noise_sigma"]=np.exp(transformed["log_noise"])
        np.savez_compressed(OUT/f"binary_row_{row}_posterior.npz",**transformed)
        truth=parameters.iloc[int(row)]
        truth_map={name:float(truth[name]) for name in
            ["period_days","phase","depth_primary","depth_ratio","duration_fraction",
             "ellipsoidal_amplitude","reflection_amplitude","noise_sigma"]}
        for name in ["period_days","phase_offset","depth_primary","depth_ratio","duration_fraction",
                     "ellipsoidal_amplitude","reflection_amplitude","noise_sigma"]:
            values=transformed[name] if name in transformed else transformed["noise_sigma"]
            q05,median,q95=np.quantile(values,[.05,.5,.95])
            truth_value=truth_map["phase"]-phase0
            truth_value=(truth_value+0.5)%1.0-0.5
            target=truth_value if name=="phase_offset" else truth_map[name]
            interval_rows.append({"simulation_row":int(row),"parameter":name,
                "q05":float(q05),"median":float(median),"q95":float(q95),
                "known_truth_or_phase_offset":float(target),
                "truth_inside_90pct_interval":bool(q05<=target<=q95)})
        med={k:float(np.median(v)) for k,v in transformed.items()}
        fitted_phase=float((phase0+med["phase_offset"])%1.0)
        pred=np.asarray(predict_jax(time_j,med["period_days"],fitted_phase,med["depth_primary"],
                    med["depth_ratio"],med["duration_fraction"],med["ellipsoidal_amplitude"],
                    med["reflection_amplitude"]),float)
        plot_rows.append((int(row),t,y,pred))
        case_summaries.append({"simulation_row":int(row),"status":status,
            "period_low":p_low,"period_high":p_high,"period_median":float(np.median(transformed["period_days"])),
            "max_r_hat":max_rhat,"min_effective_sample_size":min_ess,
            "divergences":divergences,"seconds":round(time.time()-started,1)})
        posterior_draws[int(row)]=transformed
    except Exception as exc:
        case_summaries.append({"simulation_row":int(row),"status":"failed",
            "error":f"{type(exc).__name__}: {exc}","period_low":p_low,"period_high":p_high,
            "seconds":round(time.time()-started,1)})
    pd.DataFrame(case_summaries).to_csv(OUT/"day9_5_nuts_case_diagnostics.csv",index=False)
    pd.DataFrame(interval_rows).to_csv(OUT/"day9_5_posterior_recovery_intervals.csv",index=False)
    print(f"row {row}: {case_summaries[-1]}",flush=True)


In [ ]:
import matplotlib.pyplot as plt

fig,axes=plt.subplots(len(plot_rows),1,figsize=(12,10),squeeze=False)
for ax,(row,t,y,pred) in zip(axes[:,0],plot_rows):
    ax.scatter(t,y,s=8,alpha=.35,label="synthetic observations")
    ax.plot(t,pred,color="crimson",lw=1.0,label="posterior-median fit")
    ax.set_title(f"Simulation row {row}: conditional binary posterior fit")
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Normalized flux")
    ax.legend()
fig.tight_layout()
fit_plot=OUT/"day9_5_posterior_median_fits.png"
fig.savefig(fit_plot,dpi=150)
plt.show()

case_df=pd.DataFrame(case_summaries)
interval_df=pd.DataFrame(interval_rows)
coverage={}
if not interval_df.empty:
    coverage={name:{"n":int(len(g)),"covered":int(g["truth_inside_90pct_interval"].sum()),
                    "fraction":float(g["truth_inside_90pct_interval"].mean())}
              for name,g in interval_df.groupby("parameter")}
summary={"created_utc":datetime.now(timezone.utc).isoformat(),
 "jax_backend":jax.default_backend(),"jax_devices":[str(d) for d in jax.devices()],
 "day7_binary_input":str(BINARY_DIR),"day9_4_map_input":str(map_path),
 "scope":"Three targeted synthetic binary cases; NUTS posteriors conditional on Day 9.4 period modes.",
 "configuration":{"chains":CHAINS,"warmup_per_chain":WARMUP,"samples_per_chain":SAMPLES,
                  "target_accept_prob":0.95,"dense_mass":True,"chain_method":CHAIN_METHOD},
 "screen_pass_count":int(case_df.get("status",pd.Series(dtype=str)).eq("screen_pass").sum()),
 "needs_review_or_failed_count":int(case_df.get("status",pd.Series(dtype=str)).isin(["needs_review","failed"]).sum()),
 "descriptive_90pct_interval_coverage":coverage,
 "outputs":["day9_5_nuts_case_diagnostics.csv","day9_5_posterior_recovery_intervals.csv",
            "binary_row_*_posterior.npz","day9_5_posterior_median_fits.png"],
 "limits":["Three synthetic cases only; coverage fractions are descriptive and not calibrated.",
 "Posteriors are conditional on one selected local period mode and do not include global alias uncertainty.",
 "Sampler screen-pass is not proof of model correctness or real-star validity.",
 "Known truths are used only to evaluate synthetic recovery."]}
(OUT/"day9_5_targeted_nuts_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(case_df.to_string(index=False))
print(interval_df.to_string(index=False))
print(json.dumps(summary,indent=2))
print("Outputs:",OUT)